# 11547_1 (ubiquitin): vanilla AF2 and AF2 + 2RU6 template, with the 2RU6 ensemble as grey reference

TM-aligns everything onto 2RU6 model 1, then writes the PDBs and one `.pml`.

In [ ]:
import json
import os
import urllib.request

import numpy as np
import pandas as pd
from Bio.Align import PairwiseAligner
from Bio.PDB import PDBIO, PDBParser
from Bio.PDB.Polypeptide import three_to_index, index_to_one
from tmtools import tm_align

GT_PATH = ("../../../../data/rBMRB/cluster_30p/PANAV/test.json")              # BMRB sequence only
AF2_DIR = ("../../../../data/rBMRB/cluster_30p/structures_split_filtered/test")
OUT_DIR = os.getcwd()                                   # .../structure_error_figs/11547_af2template
PDB_DIR = os.path.join(OUT_DIR, "..", "pdb_raw")        # shared with the ubiquitin notebook
os.makedirs(PDB_DIR, exist_ok=True)

PID, PDB_ID = "11547_1", "2RU6"

# AF2 inputs: PyMOL object -> (structure file, file tag, label)
AF2_INPUTS = {
    "ub_af2":      (os.path.join(AF2_DIR, f"{PID}.pdb"),                "vanilla_AF2",    "vanilla AF2"),
    "ub_af2nmr_s": (os.path.join(OUT_DIR, "11547_1_af2nmr_single.pdb"), "AF2_NMR_single", "AF2 + 2RU6 template, single seq"),
    "ub_af2nmr_m": (os.path.join(OUT_DIR, "11547_1_af2nmr_msa.pdb"),    "AF2_NMR_msa",    "AF2 + 2RU6 template, MSA"),
}
NMR_OBJ = "ub_alt_pdb"      # same object name as render_gt_error_11547_1_2RU6_vs_AF2.pml

MIN_IDENTITY = 0.95         # minimum identity over aligned positions, else refuse to write
N_MODELS = 10               # first N models of the NMR ensemble

In [ ]:
def fetch_pdb(pdb_id):
    path = os.path.join(PDB_DIR, f"{pdb_id}.pdb")
    if not os.path.exists(path):
        urllib.request.urlretrieve(f"https://files.rcsb.org/download/{pdb_id}.pdb", path)
    return path


def one_letter(res_list):
    return "".join(index_to_one(three_to_index(r.get_resname())) for r in res_list)


def load_models(path, name, n_keep=None):
    """All models (capped at n_keep), first chain, standard amino acids only (drops waters/ligands/
    hetero residues). Returns (struct, per-model residue lists), every model must carry the same
    residues as model 1."""
    struct = PDBParser(QUIET=True).get_structure(name, path)
    if n_keep is not None:
        for m in list(struct)[n_keep:]:
            struct.detach_child(m.id)
    chain_id = next(struct[0].get_chains()).id
    model_res = []
    for model in struct:
        for c in list(model):
            if c.id != chain_id:
                model.detach_child(c.id)
        chain = model[chain_id]
        for res in list(chain):
            if res.id[0] != " ":
                chain.detach_child(res.id)
        model_res.append(list(chain))
    for k, res_k in enumerate(model_res[1:], start=2):
        if one_letter(res_k) != one_letter(model_res[0]):
            raise ValueError(f"{name}: model {k} residues differ from model 1")
    return struct, model_res


_aligner = PairwiseAligner()
_aligner.mode = "global"
_aligner.match_score, _aligner.mismatch_score = 2, -1
_aligner.open_gap_score, _aligner.extend_gap_score = -5, -0.5
_aligner.end_gap_score = 0   # free terminal overhangs (tags / truncated termini)


def map_to_bmrb(seq, bmrb_seq):
    """{residue list index -> BMRB 0-based index} + list of mismatches (i, j, aa, aa)."""
    aln = _aligner.align(seq, bmrb_seq)[0]
    mapping, mismatches = {}, []
    for (s1, e1), (s2, e2) in zip(*aln.aligned):
        for i, j in zip(range(s1, e1), range(s2, e2)):
            mapping[i] = j
            if seq[i] != bmrb_seq[j]:
                mismatches.append((i, j, seq[i], bmrb_seq[j]))
    return mapping, mismatches


def ca_array(res_list):
    """CA coords (N,3) + one-letter seq of the residues that have a CA, for TM-align."""
    res = [r for r in res_list if "CA" in r]
    return np.array([r["CA"].coord for r in res], dtype=np.float64), one_letter(res)


def apply_tm(tm, atoms):
    """Apply a TM-align result (mobile = chain 1) to atoms: x -> u @ x + t."""
    for atom in atoms:
        atom.set_coord(tm.u @ atom.coord.astype(np.float64) + tm.t)

## 1. Sequence check: 2RU6 ensemble and AF2 models vs BMRB sequence

In [ ]:
gt_json = json.load(open(GT_PATH))
inv = {v: k for k, v in gt_json["entry_ID"].items()}
bmrb_seq = "".join(gt_json["sequence"][inv[PID]])


def check_seq(label, res_list):
    seq = one_letter(res_list)
    mapping, mismatches = map_to_bmrb(seq, bmrb_seq)
    identity = 1 - len(mismatches) / len(mapping)
    print(f"  {label:15s} len={len(seq)}  aligned={len(mapping)}/{len(bmrb_seq)}  identity={identity:.3f}")
    for i, j, a, b in mismatches:
        print(f"    MISMATCH {label} {res_list[i].get_resname()}{res_list[i].id[1]} ({a}) vs BMRB {j + 1} ({b})")
    if identity < MIN_IDENTITY:
        raise ValueError(f"{PID} vs {label}: identity {identity:.3f} < {MIN_IDENTITY}")
    return mapping


print(f"{PID} <-> {PDB_ID}")
pdb_struct, model_res = load_models(fetch_pdb(PDB_ID), PDB_ID, N_MODELS)
if len(model_res) != N_MODELS:
    raise ValueError(f"{PDB_ID} has only {len(model_res)} models < N_MODELS={N_MODELS}")
pdb_map = check_seq(PDB_ID, model_res[0])
af2 = {}
for obj, (path, tag, label) in AF2_INPUTS.items():
    struct, (res,) = load_models(path, obj)
    af2[obj] = dict(struct=struct, res=res, tag=tag, label=label, map=check_seq(tag, res))

## 2. Align: everything -> 2RU6 model 1

In [ ]:
ref_res = model_res[0]                                   # 2RU6 model 1 = reference, not moved
ref_xyz, ref_seq = ca_array(ref_res)
ref_ca = {pdb_map[i]: r["CA"].coord.astype(np.float64)
          for i, r in enumerate(ref_res) if i in pdb_map and "CA" in r}


def align_to_ref(res_list, mapping):
    """TM-align res_list (mobile) onto 2RU6 model 1, move its atoms, return (TM, rmsd_tmalign, rmsd_ca)."""
    xyz, seq = ca_array(res_list)
    tm = tm_align(xyz, ref_xyz, seq, ref_seq)
    apply_tm(tm, [a for r in res_list for a in r])
    ca = {mapping[i]: r["CA"].coord.astype(np.float64)
          for i, r in enumerate(res_list) if i in mapping and "CA" in r}
    d = np.array([ref_ca[j] - ca[j] for j in sorted(set(ref_ca) & set(ca))])
    return tm.tm_norm_chain2, tm.rmsd, float(np.sqrt((d ** 2).sum(1).mean()))


rows = []
for obj, a in af2.items():
    tm_s, r_tm, r_ca = align_to_ref(a["res"], a["map"])
    rows.append({"mobile": obj, "n_models": 1, "tm_score": tm_s, "rmsd_tmalign": r_tm, "rmsd_ca": r_ca})
nmr = np.array([align_to_ref(res_k, pdb_map) for res_k in model_res[1:]])
rows.append({"mobile": f"{PDB_ID} models 2-{len(model_res)} (mean)", "n_models": len(nmr),
             "tm_score": nmr[:, 0].mean(), "rmsd_tmalign": nmr[:, 1].mean(), "rmsd_ca": nmr[:, 2].mean()})
tm_df = pd.DataFrame(rows)
tm_df.to_csv(os.path.join(OUT_DIR, "11547_1_vs_2RU6_m1_tmscore.csv"), index=False)
print(tm_df.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
print(f"{PDB_ID} models 2-{len(model_res)} TM-score: " + " ".join(f"{v:.3f}" for v in nmr[:, 0]))

## 3. Write the aligned PDBs (original B-factors kept)

In [ ]:
io = PDBIO()
out_pdb = f"{PID}_{PDB_ID}_aligned.pdb"
io.set_structure(pdb_struct)
io.save(os.path.join(OUT_DIR, out_pdb))
print("wrote", out_pdb)
for obj, a in af2.items():
    a["out"] = f"{PID}_{a['tag']}_aligned.pdb"
    io.set_structure(a["struct"])
    io.save(os.path.join(OUT_DIR, a["out"]))
    print("wrote", a["out"])

## 4. PyMOL script: three AF2 inputs over the 2RU6 ensemble

In [ ]:
load_af2 = "\n".join(f"load {a['out']}, {obj}" for obj, a in af2.items())
legend = "\n".join(f"#   {obj:12s} {a['label']}" for obj, a in af2.items())
png = f"{PID}_{PDB_ID}_vs_AF2_AF2NMR.png"
B5_RESI = "68-72"

# NOTE: no ';' anywhere in the .pml (even in comments) -- PyMOL treats it as a command separator
pml = f"""delete all
bg_color white
set ray_opaque_background, 0
set orthoscopic, on

set specular, 0
set reflect, 0
set ambient, 1
set depth_cue, 0
set ray_trace_mode, 1
set ray_trace_depth_factor, 1
set ray_trace_disco_factor, 1
set ray_trace_gain, 0.005
set ray_shadow, off
set antialias, 2
set cartoon_oval_length, 1
set cartoon_rect_length, 1

########################################
# {PID} -- vanilla AF2 and AF2 + {PDB_ID} template vs {PDB_ID} ensemble (grey)
{legend}
# All structures TM-aligned onto {PDB_ID} model 1 (write_gt_error_bfactor_11547_1_af2nmr.ipynb), no fitting here
########################################
load {out_pdb}, {NMR_OBJ}_ens
{load_af2}
# one object per NMR model, grouped -> everything sits in a single frame (no multi-state movie)
split_states {NMR_OBJ}_ens, prefix={NMR_OBJ}_m
delete {NMR_OBJ}_ens
group {NMR_OBJ}, {NMR_OBJ}_m*
group ub_af2_all, {" ".join(af2)}
dss {NMR_OBJ}
dss ub_af2_all
hide everything
show cartoon, {NMR_OBJ} or ub_af2_all

# experimental ensemble: light grey
color grey80, {NMR_OBJ}

# AF2 inputs: same colors as the b5 MAE bar chart (atoms carry the object color -> uniform sticks)
set_color c_af2, [0.60, 0.60, 0.60]
set_color c_af2nmr_s, [0.380, 0.686, 0.937]
set_color c_af2nmr_m, [0.596, 0.765, 0.475]
color c_af2, ub_af2
color c_af2nmr_s, ub_af2nmr_s
color c_af2nmr_m, ub_af2nmr_m

# cartoon 0.4 transparent everywhere except b5 ({B5_RESI})
set cartoon_transparency, 0.8
set cartoon_transparency, 0.0, resi {B5_RESI}
set cartoon_oval_length, 1.0      # helix ribbon width   (default 1.35)
set cartoon_oval_width, 0.18      # helix thickness      (default 0.25)
set cartoon_rect_length, 1.0      # strand width         (default 1.40)
set cartoon_rect_width, 0.25      # strand thickness     (default 0.40)
set cartoon_loop_radius, 0.15     # loop / coil tube     (default 0.20)
set cartoon_transparency, 0.2, {NMR_OBJ} and resi {B5_RESI}

# b5: sticks (uniform object color) for the AF2 inputs, lines for the NMR ensemble, no hydrogens
# select b5, resi {B5_RESI} and not hydro
# show sticks, ub_af2_all and b5
# set stick_radius, 0.22, ub_af2_all and b5
# show lines, {NMR_OBJ} and b5
# set line_width, 1.5

hide everything, hydro
orient ub_af2
deselect
#ray 1800, 1500
#png {png}, dpi=300
"""
assert ";" not in pml
pml_path = os.path.join(OUT_DIR, f"render_{PID}_{PDB_ID}_vs_AF2_AF2NMR.pml")
with open(pml_path, "w") as fh:
    fh.write(pml)
print("wrote", os.path.basename(pml_path))